## 0. Setup & Imports

Import core dependencies: `requests` for downloading remote datasets, `tqdm` for progress bars during large file downloads, `pandas` for tabular data manipulation, and `os` for file system operations.

In [1]:
import requests
from tqdm.auto import tqdm
import pandas as pd
import os

## 0b. Mount Google Drive

Mount Google Drive to access the Elamite Lemma Dictionary and related data files stored in the FactGrid Cuneiform (AWCA) shared directory. All file paths in this notebook assume the FactGrid folder structure.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Load the Elamite Lemma Dictionary

Load the master Elamite dictionary from a multi-tab Excel file (`Elamite_Lemma-base-draft.xlsx`). Each tab corresponds to a grammatical category (ADJ, Noun, Verb, etc.), and each row is a dictionary entry with its transliteration, base lemma, morphological annotations, and period of attestation.

The `openpyxl` warning about unknown extensions is expected — the Excel file contains custom metadata that doesn't affect the data.

In [3]:
file_path = 'data/Elamite_Lemma-base-draft.xlsx'
all_sheets = pd.read_excel(file_path, sheet_name=None)


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## 2. Define Target Tabs and Columns

**Target tabs** correspond to part-of-speech categories in the dictionary:
- **ADJ, Noun, Verb** — core grammatical categories
- **PN, PN-hyp** — personal names (attested and hypothetical)
- **GN, DN** — geographical and divine names
- **Magic, other** — specialized and miscellaneous entries

**Columns to keep** include the transliteration (the romanized cuneiform reading), the base lemma, morphological decomposition (`morpheme_1/2/3`), scholarly sense annotations from Hinz & Koch (`sense_hk`) and MEGA (`sense_MEGA_qid`), and grammatical features (`POS`, `number`, `person`).

In [4]:
target_tabs = ['ADJ', 'Noun', 'Verb', 'other', 'PN', 'PN-hyp', 'GN', 'DN', 'Magic']
columns_to_keep = [
    'transliteration', 'sorting', 'period', 'base',
    'logogram', 'morpheme_1', 'morpheme_2', 'morpheme_3',
    'sense_hk', 'certainty-weight_hk', 'sense_hk_qid',
    'certainty-weight_MEGA', 'sense_MEGA_qid', 'POS', 'number', 'person'
]

## 3. Combine All Tabs into a Single DataFrame

Iterate through each target tab, tag every entry with its source tab name as a `category` column (this becomes our POS gold label), and concatenate into one unified dictionary. Only columns that actually exist in each tab are kept — not all tabs have all columns (e.g., `Magic` may lack morpheme fields).

In [5]:
combined_list = []
for name in target_tabs:
    if name in all_sheets:
        df = all_sheets[name]
        df['category'] = name
        existing_cols = [c for c in columns_to_keep if c in df.columns]
        df_filtered = df[existing_cols + ['category']]

        combined_list.append(df_filtered)

final_dictionary = pd.concat(combined_list, ignore_index=True)

### Quick Inspection

Verify the first few rows. Note the transliteration format:
- Hyphens (`-`) separate sign readings within a word: `a-a`, `h.hi-bat-tin-na`
- Dots (`.`) separate compound sign readings: `h.hi` is a single sign read as two values
- Special characters like `?`, `!`, `*` are scholarly annotations (uncertainty, collation marks)

In [6]:
final_dictionary.head(3)

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category
0,a-a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,NaN,NaN,NaN,adjective,Sg.,1st,ADJ
1,h.hi-bat-tin-na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,NaN,NaN,NaN,GN,NaN,NaN,ADJ
2,v.ha-ak-qa-man-nu-iš-ši-ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,NaN,NaN,NaN,adjective,NaN,NaN,ADJ


## 4. Download CDLI Catalogue and ATF Corpus

Download two large datasets from the Cuneiform Digital Library Initiative (CDLI):
1. **`cdli_cat.csv`** (~155MB) — the full CDLI catalogue with metadata for 350K+ cuneiform texts
2. **`cdliatf_unblocked.atf`** — the ATF (ASCII Transliteration Format) corpus

These are downloaded in streaming chunks (1024 bytes) with progress tracking. The files are saved to the CDLI subfolder on Google Drive for reuse across sessions.

> **Note:** This cell takes several minutes on the first run. The files are cached on Drive after that.

In [7]:
CHUNK = 1024
urls = ['https://github.com/cdli-gh/data/raw/master/cdli_cat.csv', 'https://github.com/cdli-gh/data/raw/master/cdliatf_unblocked.atf']
for url in urls:
    target = url.split('/')[-1]
    with requests.get(url, stream=True) as r:
        if r.status_code == 200:
            total_size = int(r.headers.get('content-length', 0))
            tqdm.write(f'Saving {url} as CDLI/{target}')
            t=tqdm(total=total_size, unit='B', unit_scale=True, desc = target)
            with open(f'data/{target}', 'wb') as f:
                for c in r.iter_content(chunk_size=CHUNK):
                    t.update(len(c))
                    f.write(c)
        else:
            print(f"{url} does not exist.")

Saving https://github.com/cdli-gh/data/raw/master/cdli_cat.csv as CDLI/cdli_cat.csv


cdli_cat.csv:   0%|          | 0.00/155M [00:00<?, ?B/s]

Saving https://github.com/cdli-gh/data/raw/master/cdliatf_unblocked.atf as CDLI/cdliatf_unblocked.atf


cdliatf_unblocked.atf:   0%|          | 0.00/86.9M [00:00<?, ?B/s]

## 5. Load the CDLI Catalogue

Load the downloaded CDLI catalogue CSV and standardize the `id_text` column to the P-number format (e.g., `P000001`) used as the universal identifier for cuneiform texts across CDLI, ORACC, and FactGrid.

In [8]:
cat = pd.read_csv('data/cdli_cat.csv', engine='python').fillna('')
cat['id_text'] = ["P" + str(no).zfill(6) for no in cat['id_text']]
cat

,accession_no,accounting_period,acquisition_history,alternative_years,ark_number,atf_source,atf_up,author,author_remarks,cdli_collation,...,seal_information,stratigraphic_level,subgenre,subgenre_remarks,surface_preservation,text_remarks,thickness,translation_source,width,object_remarks
0,,,,,21198/zz001q0dtm,"Englund, Robert K.",,CDLI,"31x61x18; Lú A 14-16.30-32.48-50; M XVIII, auf...",,...,,,Archaic Lu2 A (witness),,,,18,no translation,61,
1,,,,,21198/zz001q0dv4,"Englund, Robert K.",,CDLI,30x48x13; Lú A 13-15.23-25.?; Fundstelle wie W...,,...,,,Archaic Lu2 A (witness),,,,13,no translation,48,
2,,,,,21198/zz001q0dwn,"Englund, Robert K.",,"Englund, Robert K. & Nissen, Hans J.","42x53x19; Vocabulary 9; Qa XVI,2, unter der Ab...",,...,,,Archaic Vocabulary (witness),Text category: 15-09; Foreign ID: LVO 9,,,19,no translation,53,
3,,,,,21198/zz001q0dx5,"Englund, Robert K.",,CDLI,26x23x23; Lú A 9-10.?.?; Fundstelle wie W 9123...,,...,,,Archaic Lu2 A (witness),,,,23,no translation,23,
4,,,,,21198/zz001q0dzp,"Englund, Robert K.",,CDLI,"29x36x20; Lú A Vorläufer; Qa XVI,2, unter der ...",,...,,,Archaic Lu2 A (witness),,,,20,no translation,36,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353278,,,,,,no atf,,"Fahad, Saad Salman & Al-Hussainy, Abbas A.",,,...,,,,,,,,no translation,,
353279,,,,,,no atf,,"Postgate, J. Nicholas",,,...,,,,,,,20,no translation,34,
353280,,,,,,no atf,,"Postgate, J. Nicholas",,,...,,,,,,,20,no translation,34,
353281,,,"purchased from M. Gejou, Paris, in the summer ...",,,no atf,,"Grant, Elihu",,,...,,,,,,,,no translation,,


## 6. Load Sign List 1: Nuolenna

Load the first cuneiform sign list from the [Nuolenna repository](https://github.com/situx/Nuolenna). This JSON file maps sign reading names (e.g., `a`, `ab`, `an`) to their Unicode cuneiform code points (e.g., `𒀀`, `𒀊`, `𒀭`).

The index of the JSON becomes the sign name; the single value column is the Unicode character.

In [9]:
# Reading in the signlist from json format
sign_list = pd.read_json('https://raw.githubusercontent.com/situx/Nuolenna/master/sign_list.json', orient='index')
# name the columns, cleaning, rearranging
sign_list.columns = ['unicode']
form = sign_list.index.tolist()
sign_list['sign'] = form
desired_order = ['sign', 'unicode']
sign_list = sign_list[desired_order]
sign_list = sign_list.reset_index(drop=True)

## 7. Load Sign List 2: Akkademia

Load the second sign list from the [Akkademia project](https://github.com/gaigutherz/Akkademia). This CSV contains 14,241 sign-to-Unicode mappings, including many variant readings and numeric subscript forms (e.g., `u2`, `aia2`).

The Akkademia list is more comprehensive for variant readings, while Nuolenna has better coverage of rare signs. We merge both to maximize coverage.

In [10]:
akkademia = pd.read_csv('https://raw.githubusercontent.com/gaigutherz/Akkademia/master/cuneiform_to_unicode_fixed.csv')
akkademia

,sign,unicode
0,ʾu₄,𒀀
1,'u4,𒀀
2,a,𒀀
3,aia₂,𒀀
4,aia2,𒀀
...,...,...
14236,/,𒑰
14237,:,𒑱
14238,":""",𒑲
14239,:.,𒑳


## 8. Merge the Two Sign Lists

Perform an outer merge on both `sign` and `unicode` columns to combine the Nuolenna and Akkademia sign lists. The `_merge` indicator column tracks which source each mapping came from:
- `both` — the mapping exists in both lists (confirmed)
- `sign_list` — only in Nuolenna
- `akkademia` — only in Akkademia

Non-overlapping rows are identified for quality inspection.

In [11]:
concatenated = pd.merge(sign_list, akkademia, on=['sign', 'unicode'], how='outer', indicator=True)
concatenated['_merge'] = concatenated['_merge'].replace({'left_only': 'sign_list', 'right_only': 'akkademia'})
non_overlapping_rows = concatenated[concatenated['_merge'] != 'both']

/tmp/ipykernel_26470/868275122.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  concatenated['_merge'] = concatenated['_merge'].replace({'left_only': 'sign_list', 'right_only': 'akkademia'})


## 9. Create Working Copy for Unicode Conversion

Create a working copy of the combined dictionary (`to_unicode`) that will be progressively transformed through the cleaning pipeline. The original data in `final_dictionary` remains untouched.

In [12]:
to_unicode = final_dictionary.copy()
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category
0,a-a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,NaN,NaN,NaN,adjective,Sg.,1st,ADJ
1,h.hi-bat-tin-na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,NaN,NaN,NaN,GN,NaN,NaN,ADJ
2,v.ha-ak-qa-man-nu-iš-ši-ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,NaN,NaN,NaN,adjective,NaN,NaN,ADJ
3,am-mín-nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,NaN,NaN,NaN,adjective,Sg.,1st,ADJ
4,te-man?-na?-na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,NaN,NaN,NaN,adjective,NaN,NaN,ADJ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti-la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN
15597,til-la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN
15598,ú-zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN
15599,ur-pa-gu-b[a-ak],urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN


## 10. Load Manual Sign Corrections

Load two CSV files containing manually curated sign mappings for signs that neither Nuolenna nor Akkademia could resolve. These were created by one of the authors (referred to as 'AA' in filenames) through manual inspection of the unmatched signs from earlier runs.

- **`unmatchednew_AAedit`**: AA's corrections mapping unmatched sign readings to their Unicode equivalents
- **`unmatchednew - solonew.csv`**: Additional solo corrections

The `value` column in the second file is stripped of brackets and quotes (`[]' `) to clean up formatting artifacts.

All three dictionaries are merged in priority order: base sign list → manual corrections 1 → manual corrections 2. Later entries override earlier ones for the same key.

The test `unicode_dict['h']` verifies that the manual mapping for the sign `h` (→ `𒀸`) was loaded correctly — this sign is common in Elamite but was missing from the automated lists.

In [13]:
unmatched = pd.read_csv('data/unmatchednew_AAedit - unmatchednew.csv')
unmatched2 = pd.read_csv('data/unmatchednew - solonew.csv')
unmatched = unmatched[['unmatched_sign','use']].dropna()
unmatched2 = unmatched2[['value', 'SIGN']].dropna()
manual_dict = dict(zip(unmatched['unmatched_sign'], unmatched['use']))
manual_dict2= dict(zip(unmatched2['value'].str.strip("[]' "), unmatched2['SIGN']))
unicode_dict = dict(zip(concatenated['sign'], concatenated['unicode']))
unicode_dict.update(manual_dict)
unicode_dict.update(manual_dict2)
unicode_dict['h']

'𒀸'

## 11. Preserve Original Transliteration & Check for Placeholder Signs

Save the original transliteration to a new column (`transliteration original`) before any modifications — this is essential for later matching against the Nasu corpus and for traceability.

Check whether the placeholder sign `……` (used in cuneiform transliteration to mark intentional gaps or breaks in the text) appears in any entry. This determines how the tilde-removal step (Cell 17) should handle the placeholder.

In [14]:
#1 check sign

to_unicode['transliteration original'] = to_unicode['transliteration']

to_unicode['transliteration'] = to_unicode['transliteration'].astype(str).fillna('')


# Sign to check for existence
sign_to_check = '……'

# Method 1: Using the "any" method with a lambda function
exists = to_unicode.iloc[:, 0].apply(lambda x: sign_to_check in x).any()

# Method 2: Using the "str.contains" method
exists = to_unicode.iloc[:, 0].str.contains(sign_to_check).any()

# Check if the sign exists in the second column
if exists:
    print(f"The sign '{sign_to_check}' exists in the first column.")
else:
    print(f"The sign '{sign_to_check}' does not exist in the first column.")

The sign '……' does not exist in the first column.


## 12. First Cleaning Step: Hyphens → Spaces

Replace hyphens with spaces. In cuneiform transliteration, hyphens separate sign readings within a word:
- `h.hi-bat-tin-na` → `h.hi bat tin na`
- `a-a` → `a a`

After this step, each space-separated token is an individual sign reading that can be looked up in the Unicode dictionary.

In [15]:
#3 replace hyphen with space
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('-', ' ')
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,a-a
1,h.hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,NaN,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na
2,v.ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu
4,te man? na? na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ti-la
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,til-la
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ú-zi
15599,ur pa gu b[a ak],urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak]


## 13. Remove Scholarly Annotation Characters

Remove all editorial and scholarly annotation marks that are not part of the actual sign readings:
- `_` — determinative marker in some conventions
- `[`, `]` — restoration markers (damaged text reconstructed by scholars)
- `*` — collation mark (sign verified against the original tablet)
- `!` — correction mark
- `?` — uncertain reading
- `/`, `,`, `:`, `;`, `^`, `` ` `` — various editorial marks

**Critical:** The dot (`.`) is replaced with a **space**, not removed. This is because dots in cuneiform transliteration separate compound sign readings: `h.hi` is a compound sign that should become two tokens `h hi` for individual lookup in the Unicode dictionary.

In [16]:
#4. Remove: _, [, ], *, !, ?, /, , :, ;, ^, `, (no spaces added)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('_', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('[', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(']', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('*', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('!', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('?', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('/', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(',', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(':', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(';', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('^', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('`', '')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('.', ' ', regex=False)

# _, [, ], *, !, ?, /, , :, ;, ^, `, (no spaces added)
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,a-a
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,NaN,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ti-la
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,til-la
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ú-zi
15599,ur pa gu ba ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak]


## 14. Lowercase All Sign Readings

Convert all sign readings to lowercase. In standard Assyriology convention, uppercase readings (e.g., `GIŠ`, `DINGIR`) indicate Sumerograms (Sumerian logograms used in Elamite text). Lowercasing normalizes these for consistent dictionary lookup, since the sign lists use lowercase keys.

In [17]:
#8 lower case
to_unicode['transliteration'] = to_unicode['transliteration'].str.lower()
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,a-a
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,NaN,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ti-la
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,til-la
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ú-zi
15599,ur pa gu ba ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak]


## 15. Remove Tilde Annotations and Unreadable Signs

Two cleaning operations:

1. **Tilde removal**: The tilde (`~`) in transliteration marks variant sign forms (e.g., `~a`, `~b`, `~c`, `~t`). The regex `~.*?……` removes the tilde and everything after it up to the next `……` placeholder. This prevents variant annotations from corrupting the sign readings.

2. **X removal**: The character `X` represents unreadable or completely damaged signs on the tablet. These carry no linguistic information and are removed entirely.

In [18]:
# additional rules:
# 1 Remove / delete the following: erased, ~a, ~b, ~c, ~t, (anything immediately following the tilde ~)
# Function to remove "~" and anything following it until "……" appears
def remove_tilde_and_following(df_column):
    return df_column.str.replace(r'~.*?……', '……')

# Apply the function to the desired column
to_unicode['transliteration'] = remove_tilde_and_following(to_unicode['transliteration'])


# if we were to remove "X"
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('X', '')
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,a-a
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,NaN,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ti-la
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,til-la
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ú-zi
15599,ur pa gu ba ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak]


## 16. Restore Placeholder Signs to Spaces

Convert the `……` placeholder sign back to a regular space. The placeholder was used as a safe boundary during tilde removal (Cell 17) to prevent the regex from consuming too much text. Now that tilde variants are cleaned, the placeholder is no longer needed.

In [19]:
#9 change …… sign back to space
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('……', ' ')
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,a-a
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,NaN,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ti-la
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,til-la
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ú-zi
15599,ur pa gu ba ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak]


## 17. First-Pass Unicode Conversion

The core conversion step. For each entry, split the cleaned transliteration into individual sign tokens and look each one up in the merged Unicode dictionary:
- If a mapping exists → replace with the Unicode cuneiform character
- If no mapping exists → keep the original Latin token (will be addressed in the second pass)

Example: `h hi bat tin na` → `𒀸 𒄭 𒁁 𒁷 𒈾`

The commented-out code shows earlier iterations that attempted additional character stripping and double-pass conversion — these were superseded by the second-pass approach in Cell 27.

In [20]:
#10 convert to sign
# Create a dictionary from dataframe A mapping signs to unicode
def replace_with_unicode(text):
    unicode_values = []
    for sign in text.split():
        unicode = unicode_dict.get(sign)
        if unicode is not None:
            unicode_values.append(unicode)
        else:
            unicode_values.append(sign)  # Keep the sign if no corresponding unicode value found
    return ' '.join(unicode_values)

to_unicode['unicode'] = to_unicode['transliteration'].apply(replace_with_unicode)

# Second time filtering speical characters
# def remove_special_characters(text):
#     cleaned_text = re.sub(r'[!?#.\[\]]', '', text)
#     return cleaned_text

# Apply the function to the 'sentences' column
# to_unicode['unicode'] = to_unicode['unicode'].apply(remove_special_characters)

# to_unicode['unicode'] = to_unicode['unicode'].apply(replace_with_unicode)
to_unicode

# if we were to remove "X"
# to_unicode['unicode'] = to_unicode['unicode'].str.replace('X', '')
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original,unicode
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,a-a,𒀀 𒀀
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,NaN,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na,𒀸 𒄭 𒁁 𒁷 𒈾
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya,𒁹 𒄩 𒀝 𒋡 𒌋𒌋 𒉡 𒅖 𒅆 𒉿
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,NaN,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu,𒄠 𒊩 𒉡
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,NaN,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na,𒋼 𒌋𒌋 𒈾 𒈾
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ti-la,𒋾 𒆷
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,til-la,𒌀 𒆷
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ú-zi,ú 𒍣
15599,ur pa gu ba ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak],𒌨 𒉺 𒄖 𒁀 𒀝


## 18. Identify Failed Conversions (First Pass)

Find all entries that still contain Latin alphabet characters after the first Unicode conversion pass. This uses a regex to extract any remaining alphabetical tokens from the `unicode` column.

The result is merged back with the full dataframe to create a report of failed conversions with their original context (sorting, period, base lemma, etc.). This report was used to create the manual correction files loaded in Cell 12.

In [21]:
# this is the code for sorting out the code that did not get converted to unicode signs
import re

alphabetical = r'\b[A-Za-z]+\b'
alphabetical_list = to_unicode['unicode'].apply(lambda x: re.findall(alphabetical, x))
alphabetical_list

series_data_reset = alphabetical_list.reset_index()

series_data_reset.columns = ['Row Number', 'words']

temp = series_data_reset.melt(id_vars='Row Number', value_vars='words')

filtered_df = temp[['value']]

failed_converted_words = filtered_df.rename(columns={'value': 'words'})

failed_converted_words = filtered_df[filtered_df['value'].apply(lambda x: len(x) > 0)]

failed_cw_formatted = failed_converted_words.copy()
failed_cw_formatted = pd.merge(failed_cw_formatted, to_unicode, left_index=True, right_index=True)

failed_converted_words = pd.merge(failed_converted_words, to_unicode, left_index=True, right_index=True).drop(["transliteration","unicode"],  axis=1)

### Inspect Failed Conversions

Display the failed conversion report. Each row shows a sign that could not be automatically mapped to Unicode, along with its dictionary context. These are the signs that needed manual curation by one of the authors.

In [22]:
failed_converted_words

,value,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,sense_hk_qid,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original
560,[ut],s~uutkime,mE,šutki.me,NaN,-me;3sg.inan,NaN,NaN,bei Nacht,1,NaN,NaN,NaN,adverb,NaN,NaN,ADJ,šu-<ut>-ki-me
561,[ut],s~uutkime,mE,šut,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,noun,NaN,NaN,ADJ,šu-<ut>-ki-me
562,[ut],s~uutkime,mE,ki,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,noun,NaN,NaN,ADJ,šu-<ut>-ki-me
596,[za],zabarya,mE,zubar,NaN,-ya;adj,NaN,NaN,aus Kupfer,1,NaN,NaN,NaN,noun,NaN,NaN,ADJ,<za>-bar-ya
681,[X],abuuls~ati,mE,NaN,NaN,NaN,NaN,NaN,0,0.5,NaN,NaN,NaN,noun,SG,1.0,Noun,a-bu-ul.ša-ti-[x]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14692,[X],laba,achE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GN,NaN,NaN,GN,h.la-ba?-x
14807,[X],maunais~,achE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GN,NaN,NaN,GN,h.ma-u-na-[x]-iš
14920,[X],raku,achE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GN,NaN,NaN,GN,h.ra-[ku?-x]
15206,[X],uzzi,mE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GN,NaN,NaN,GN,h.uz-z[i-x]


### Export Failed Conversions

Save the failed conversion report to CSV for offline manual correction. This file (`unmatchednewfinal.csv`) is the input for the next round of manual sign mapping.

In [23]:
failed_converted_words.to_csv('data/unmatchednewfinal.csv')

## 19. Flag Entries with Remaining Latin Characters

Create two diagnostic columns:
- **`roman`** (boolean): `True` if the Unicode column still contains any Latin alphabet characters (i.e., incomplete conversion)
- **`count`** (integer): the number of unconverted Latin tokens remaining

Uses the `regex` library's `\p{Latin}` Unicode property class, which correctly handles accented Latin characters (e.g., `š`, `ṭ`, `ú`) that `re.ASCII` would miss.

In [24]:
import regex as re
# Function to check if a word has Roman alphabet
def has_roman_alphabet(word):
    return bool(re.search(r'\p{Latin}', word))

# Function to count the number of words with Roman alphabet in a sentence
def count_roman_words(sentence):
    words = sentence.split()
    return sum(has_roman_alphabet(word) for word in words)

to_unicode['roman'] = to_unicode['unicode'].apply(lambda x: any(has_roman_alphabet(word) for word in x.split()))
to_unicode['count'] = to_unicode['unicode'].apply(count_roman_words)

to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,...,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original,unicode,roman,count
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,...,NaN,NaN,adjective,Sg.,1st,ADJ,a-a,𒀀 𒀀,False,0
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,...,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na,𒀸 𒄭 𒁁 𒁷 𒈾,False,0
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,...,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya,𒁹 𒄩 𒀝 𒋡 𒌋𒌋 𒉡 𒅖 𒅆 𒉿,False,0
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,...,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu,𒄠 𒊩 𒉡,False,0
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,...,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na,𒋼 𒌋𒌋 𒈾 𒈾,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ti-la,𒋾 𒆷,False,0
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,til-la,𒌀 𒆷,False,0
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ú-zi,ú 𒍣,True,1
15599,ur pa gu ba ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak],𒌨 𒉺 𒄖 𒁀 𒀝,False,0


### Create Comparison View

Extract the cleaned transliteration alongside the original for side-by-side comparison. This is useful for verifying that the cleaning pipeline didn't corrupt any entries.

In [25]:
failed_cw_formatted = to_unicode[['transliteration', 'transliteration original']]

### Display Comparison

Show the side-by-side comparison. Key transformations visible:
- `a-a` → `a a` (hyphens to spaces)
- `h.hi-bat-tin-na` → `h hi bat tin na` (dots and hyphens to spaces)
- `v.ha-ak-qa-man-nu-iš-ši-ya` → `v ha ak qa man nu iš ši ya` (compound signs split)

In [26]:
failed_cw_formatted

,transliteration,transliteration original
0,a a,a-a
1,h hi bat tin na,h.hi-bat-tin-na
2,v ha ak qa man nu iš ši ya,v.ha-ak-qa-man-nu-iš-ši-ya
3,am mín nu,am-mín-nu
4,te man na na,te-man?-na?-na
...,...,...
15596,ti la,ti-la
15597,til la,til-la
15598,ú zi,ú-zi
15599,ur pa gu ba ak,ur-pa-gu-b[a-ak]


### Full DataFrame Inspection

Display the complete working dataframe with all columns including the Unicode conversion results and the roman/count diagnostic flags.

In [27]:
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,...,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original,unicode,roman,count
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,...,NaN,NaN,adjective,Sg.,1st,ADJ,a-a,𒀀 𒀀,False,0
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,...,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na,𒀸 𒄭 𒁁 𒁷 𒈾,False,0
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,...,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya,𒁹 𒄩 𒀝 𒋡 𒌋𒌋 𒉡 𒅖 𒅆 𒉿,False,0
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,...,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu,𒄠 𒊩 𒉡,False,0
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,...,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na,𒋼 𒌋𒌋 𒈾 𒈾,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ti-la,𒋾 𒆷,False,0
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,til-la,𒌀 𒆷,False,0
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ú-zi,ú 𒍣,True,1
15599,ur pa gu ba ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak],𒌨 𒉺 𒄖 𒁀 𒀝,False,0


## 20. Second-Pass Unicode Conversion (Critical)

This is the most important step in the pipeline. After the first pass, many entries still contain unconverted Latin tokens due to edge cases the simple dictionary lookup missed. This second pass applies more aggressive normalization before retrying:

**`normalize_for_unmatched_pass(s)`:**
1. Splits determinative markers `(md)` into separate `m d` tokens
2. Replaces all remaining punctuation with spaces
3. Converts hyphens/dashes to spaces
4. **Splits digit-letter boundaries**: `u2a` → `u2 a`, ensuring numeric sign indices are separated from the next sign reading (e.g., `u2` is the sign reading "u-two", distinct from `u`)
5. Collapses multiple spaces

**`final_unmatched_conversion_pass(unicode_text)`:**
1. Applies the normalization above
2. Special-cases `md` → `m` + `d` (the masculine determinative compound)
3. Retries the Unicode dictionary lookup on each re-tokenized element
4. Tracks any signs that still can't be converted

**Result:** Reduces unmatched signs from hundreds to just 23 (later 20 after additional corrections). This function is the key to achieving the 85.8% conversion rate.

In [28]:
import regex as re
import pandas as pd

unicode_dict = dict(zip(concatenated["sign"].astype(str), concatenated["unicode"].astype(str)))

def normalize_for_unmatched_pass(s):
    if pd.isna(s):
        return ""
    s = str(s)
    #split m and d
    s = re.sub(r"\(\s*md\s*\)", " m d ", s)
    s = re.sub(r"[.,:;!?()\[\]{}<>\"“”‘’/\\|*^`~]", " ", s)
    s = re.sub(r"[-–—]", " ", s)

    s = re.sub(r"([A-Za-z])(\d)([A-Za-z])", r"\1\2 \3", s)
    s = re.sub(r"(\d)([A-Za-z])", r"\1 \2", s)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def final_unmatched_conversion_pass(unicode_text):
    txt = normalize_for_unmatched_pass(unicode_text)
    tokens = txt.split()

    out = []
    still_unmatched = []
    for tok in tokens:
        if tok == "md" and ("m" in unicode_dict) and ("d" in unicode_dict) and ("md" not in unicode_dict):
          out.append(unicode_dict["m"])
          out.append(unicode_dict["d"])
          continue

        if tok in unicode_dict:
            out.append(unicode_dict[tok])
        else:
            out.append(tok)
            if re.search(r'\p{Latin}', tok):
                still_unmatched.append(tok)

    return " ".join(out), still_unmatched

tmp = to_unicode["unicode"].apply(final_unmatched_conversion_pass)
to_unicode["transliteration"] = to_unicode["transliteration original"].apply(normalize_for_unmatched_pass)
to_unicode["roman"] = tmp.apply(lambda x: len(x[1]) > 0)
to_unicode["count"] = tmp.apply(lambda x: len(x[1]))
to_unicode["unicode"] = tmp.apply(lambda x: x[0])

all_unmatched = []
for lst in tmp.apply(lambda x: x[1]):
    all_unmatched.extend(lst)

failed_converted_words = pd.DataFrame(sorted(set(all_unmatched)), columns=["unmatched_sign"])

print("Final unmatched count:", len(failed_converted_words))
failed_converted_words


Final unmatched count: 23


,unmatched_sign
0,X
1,h
2,hh
3,lugàl
4,nap4
5,pi+pír
6,rák
7,sap6
8,sir99
9,uballiṭ


### Verify Second-Pass Results

Display the dataframe after the second conversion pass. The `unicode` column should now contain mostly cuneiform characters, with the `roman` flag showing which entries still have unconverted tokens.

In [29]:
to_unicode

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,...,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original,unicode,roman,count
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,...,NaN,NaN,adjective,Sg.,1st,ADJ,a-a,𒀀 𒀀,False,0
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,...,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na,𒀸 𒄭 𒁁 𒁷 𒈾,False,0
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,...,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya,𒁹 𒄩 𒀝 𒋡 𒌋𒌋 𒉡 𒅖 𒅆 𒉿,False,0
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,...,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu,𒄠 𒊩 𒉡,False,0
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,...,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na,𒋼 𒌋𒌋 𒈾 𒈾,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ti-la,𒋾 𒆷,False,0
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,til-la,𒌀 𒆷,False,0
15598,ú zi,uzi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ú-zi,ú 𒍣,True,1
15599,ur pa gu b a ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak],𒌨 𒉺 𒄖 𒁀 𒀝,False,0


## 21. Filter to Fully Converted Entries

Create the clean subset by filtering to entries where `roman == False` (no remaining Latin characters in the Unicode column). These are the entries that were fully converted and are suitable for downstream analysis.

This produces the `to_unicode_clean` dataframe — approximately 13,300 entries out of 15,600 total (85.8% conversion rate).

In [30]:
to_unicode_clean = to_unicode[to_unicode['roman'] == False]
to_unicode_clean

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,...,certainty-weight_MEGA,sense_MEGA_qid,POS,number,person,category,transliteration original,unicode,roman,count
0,a a,aa,oE; nE,ai,NaN,NaN,NaN,NaN,"wohl, gut, gern",0.5,...,NaN,NaN,adjective,Sg.,1st,ADJ,a-a,𒀀 𒀀,False,0
1,h hi bat tin na,hibattinna,achE,Hipat,NaN,-na;postp.gen,NaN,NaN,"Adjektivbildung zu dem Ortsnamen h.hi-ba-at,",1,...,NaN,NaN,GN,NaN,NaN,ADJ,h.hi-bat-tin-na,𒀸 𒄭 𒁁 𒁷 𒈾,False,0
2,v ha ak qa man nu iš ši ya,haakqamannuis~s~iya,achE,hakamanušiya,NaN,-š;op.loan,-ya;adj,NaN,achämenidisch,1,...,NaN,NaN,adjective,NaN,NaN,ADJ,v.ha-ak-qa-man-nu-iš-ši-ya,𒁹 𒄩 𒀝 𒋡 𒌋𒌋 𒉡 𒅖 𒅆 𒉿,False,0
3,am mín nu,amminnu,achE,aminu,NaN,NaN,NaN,NaN,selbig,1,...,NaN,NaN,adjective,Sg.,1st,ADJ,am-mín-nu,𒄠 𒊩 𒉡,False,0
4,te man na na,demannana,achE,tema (?),NaN,-na;postp.gen,NaN,NaN,abendlich,0.5,...,NaN,NaN,adjective,NaN,NaN,ADJ,te-man?-na?-na,𒋼 𒌋𒌋 𒈾 𒈾,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15595,šu zi ga,s~uziga,oAkk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,šu-zi-ga,𒋗 𒍣 𒂵,False,0
15596,ti la,dila,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ti-la,𒋾 𒆷,False,0
15597,til la,dilla,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,til-la,𒌀 𒆷,False,0
15599,ur pa gu b a ak,urpagubaak,Akk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,DN,NaN,NaN,DN,ur-pa-gu-b[a-ak],𒌨 𒉺 𒄖 𒁀 𒀝,False,0


## 22. Load the Utu-našu (Nasu) Corpus

Load the word-level annotated texts from the Untash-Napirisha inscriptions. This is a separate corpus of actual Elamite texts (not a dictionary), with each row representing one word token annotated with its document ID (`id_text`), word ID (`id_word`), transliteration, and lemma.

In [31]:
file_path = 'data/UnTN-Nasu texts Word-level.csv'
comparison_df = pd.read_csv(file_path)


### Inspect Nasu Transliterations

Display the transliteration column to verify the format. Note the presence of determinatives in parentheses: `(m)` for masculine, `(md)` for masculine + divine, `(d)` for divine. These follow standard Assyriological convention.

In [32]:
comparison_df['transliteration']

,transliteration
0,u2
1,(m)un-taš-dingir-gal
2,ša-ak
3,(md)hu-ban-nu-me-na-gi
4,su-un-ki-ik
...,...
2577,(d)pi-el-ti-ya
2578,na-pir2-ri-ša-ar-ra-pi
2579,uk-ku-pi
2580,ip


## 23. Match Dictionary Entries Against the Nasu Corpus

Check how many dictionary entries have their original transliteration form attested in the Nasu corpus. This measures the overlap between the lexicon and the actual text corpus.

**Result:** 222 out of 13,316 clean dictionary entries are directly attested in the Nasu texts. The low overlap is expected — the dictionary contains entries from all periods and text types, while the Nasu corpus is a single collection of royal inscriptions.

In [33]:
to_unicode_clean['is_match'] = to_unicode_clean['transliteration original'].isin(comparison_df['transliteration'])


print(to_unicode_clean['is_match'].value_counts())

is_match
False    13094
True       222
Name: count, dtype: int64


/tmp/ipykernel_26470/3081551588.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  to_unicode_clean['is_match'] = to_unicode_clean['transliteration original'].isin(comparison_df['transliteration'])


### Inspect the Nasu Corpus Structure

Display the full Nasu dataframe to see all available columns: `id_text` (document identifier), `id_word` (unique word identifier), `transliteration`, and `lemma` (scholarly lemmatization).

In [34]:
comparison_df

,id_text,id_word,transliteration,lemma,morpheme1,morpheme2,morpheme3,base suffixes,base suffixes2
0,UntN TZ 1,UntN_TZ_1_00001,u2,u,NaN,NaN,NaN,NaN,NaN
1,UntN TZ 1,UntN_TZ_1_00002,(m)un-taš-dingir-gal,Untash-Napirisha,NaN,NaN,NaN,NaN,NaN
2,UntN TZ 1,UntN_TZ_1_00003,ša-ak,šak,NaN,NaN,NaN,NaN,NaN
3,UntN TZ 1,UntN_TZ_1_00004,(md)hu-ban-nu-me-na-gi,Humban-umena,1sg.an,NaN,NaN,NaN,NaN
4,UntN TZ 1,UntN_TZ_1_00005,su-un-ki-ik,sunki,1sg.an,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2577,Nasu,Nasu_00038,(d)pi-el-ti-ya,Peltiya,NaN,NaN,NaN,NaN,NaN
2578,Nasu,Nasu_00039,na-pir2-ri-ša-ar-ra-pi,"nap, riša",3pl.an,NaN,NaN,NaN,NaN
2579,Nasu,Nasu_00040,uk-ku-pi,uku,3pl.an,NaN,NaN,NaN,NaN
2580,Nasu,Nasu_00041,ip,i,3pl.an,NaN,NaN,NaN,NaN


### View Matched Dictionary Entries

Display the 222 dictionary entries that are directly attested in the Nasu corpus. These entries have exact transliteration matches and can serve as validated anchors for cross-referencing between the dictionary and the corpus.

In [35]:
to_unicode_clean[to_unicode_clean['is_match'] == True]

,transliteration,sorting,period,base,logogram,morpheme_1,morpheme_2,morpheme_3,sense_hk,certainty-weight_hk,...,sense_MEGA_qid,POS,number,person,category,transliteration original,unicode,roman,count,is_match
28,ap pu ki i,appukii,mE,apuki,NaN,NaN,NaN,NaN,ewig,0.5,...,NaN,noun,NaN,NaN,ADJ,ap-pu-ki-i,𒀊 𒁍 𒆠 𒄿,False,0,True
46,ri e ru um ma,rieruumma,mE,reru,NaN,-ma;postp.loc,NaN,NaN,im Getreidefeld,0.5,...,NaN,noun,NaN,NaN,ADJ,ri-e-ru-um-ma,𒊑 𒂊 𒊒 𒌝 𒈠,False,0,True
112,pi tu4 ma,bituma,mE,pitu,NaN,-ma;postp.loc,NaN,NaN,innen,1,...,NaN,noun,NaN,NaN,ADJ,pi-tu4-ma,𒉿 𒌈 𒈠,False,0,True
114,pi tu4 um ma,bituumma,mE,pitu,NaN,-ma;postp.loc,NaN,NaN,innen,1,...,NaN,noun,NaN,NaN,ADJ,pi-tu4-um-ma,𒉿 𒌈 𒌝 𒈠,False,0,True
229,ku du um ma,guduumma,mE,kidu,NaN,-ma;postp.loc,NaN,NaN,äußer,1,...,NaN,noun,NaN,NaN,ADJ,ku-du-um-ma,𒆪 𒁺 𒌝 𒈠,False,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7383,ur,ur,mE; nE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,other,SG,1.0,other,ur,𒌨,False,0,True
8279,ha ap ši,haaps~i,oE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,PN,NaN,NaN,PN,ha-ap-ši,𒄩 𒀊 𒅆,False,0,True
13621,an za an,anzaan,"oE, mE, nE, achE",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,GN,NaN,NaN,GN,an-za-an,𒀭 𒍝 𒀭,False,0,True
14059,šu šu un,s~us~uun,"mE, nE",NaN,NaN,NaN,NaN,NaN,Susa,NaN,...,NaN,GN,NaN,NaN,GN,šu-šu-un,𒋗 𒋗 𒌦,False,0,True


## 24. Export the Clean Unicode Dictionary

Save the fully converted Unicode dictionary to CSV (`Elamite-Lemma-Base-Unicode.csv`) for use in downstream analyses and experiments. This is the primary output of the preprocessing pipeline.

In [36]:
to_unicode_clean.to_csv('data/Elamite-Lemma-Base-Unicode.csv')

## 25. Load Pre-Converted Nasu Unicode Data

Load a previously converted version of the Nasu corpus (`to_unicode_nasu_clean.csv`) that includes a `unicode` column. This file was produced in an earlier processing run and serves as a comparison reference.

In [37]:
file_path = 'data/to_unicode_nasu_clean.csv'
comparison_df2 = pd.read_csv(file_path)

### Inspect Pre-Converted Nasu Data

Display the pre-converted Nasu dataframe with its Unicode column, morpheme annotations, and other fields.

In [38]:
comparison_df2

,Unnamed: 0,id_text,lemma,morpheme1,morpheme2,morpheme3,base suffixes,base suffixes2,id_word,transliteration,transliteration original,unicode,roman,count
0,0,UntN TZ 1,u,NaN,NaN,NaN,NaN,NaN,UntN_TZ_1_00001,u2,u2,𒌑,False,0
1,1,UntN TZ 1,Untash-Napirisha,NaN,NaN,NaN,NaN,NaN,UntN_TZ_1_00002,m un taš dingir gal,(m)un-taš-dingir-gal,𒁹 𒌦 𒌨 𒀭 𒃲,False,0
2,2,UntN TZ 1,šak,NaN,NaN,NaN,NaN,NaN,UntN_TZ_1_00003,ša ak,ša-ak,𒊭 𒀝,False,0
3,3,UntN TZ 1,Humban-umena,1sg.an,NaN,NaN,NaN,NaN,UntN_TZ_1_00004,m d hu ban nu me na gi,(md)hu-ban-nu-me-na-gi,𒁹 𒀭 𒄷 𒉼 𒉡 𒈨 𒈾 𒄀,False,0
4,4,UntN TZ 1,sunki,1sg.an,NaN,NaN,NaN,NaN,UntN_TZ_1_00005,su un ki ik,su-un-ki-ik,𒋢 𒌦 𒆠 𒅅,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2565,2577,Nasu,Peltiya,NaN,NaN,NaN,NaN,NaN,Nasu_00038,d pi el ti ya,(d)pi-el-ti-ya,𒀭 𒉿 𒂖 𒋾 𒉿,False,0
2566,2578,Nasu,"nap, riša",3pl.an,NaN,NaN,NaN,NaN,Nasu_00039,na pir2 ri ša ar ra pi,na-pir2-ri-ša-ar-ra-pi,𒈾 𒂟 𒊑 𒊭 𒅈 𒊏 𒉿,False,0
2567,2579,Nasu,uku,3pl.an,NaN,NaN,NaN,NaN,Nasu_00040,uk ku pi,uk-ku-pi,𒊌 𒆪 𒉿,False,0
2568,2580,Nasu,i,3pl.an,NaN,NaN,NaN,NaN,Nasu_00041,ip,ip,𒅁,False,0


## 26. Cross-Reference: Unicode-Level Matching

Count how many dictionary entries have their Unicode representation attested in the pre-converted Nasu corpus. This is a stricter match than transliteration-level (Cell 32) because it compares at the cuneiform sign level.

**Result:** 301 matches at the Unicode level vs. 222 at the transliteration level — the Unicode matching finds additional overlaps because the normalization collapses variant transliterations that map to the same cuneiform signs.

In [39]:
sum(to_unicode_clean['unicode'].isin(comparison_df2['unicode']))

301